In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy as stats
import seaborn as sns
import numpy as np

# 출력 짤림 방지
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# 데이터 불러오기
df = pd.read_csv('./data/Courses.csv', parse_dates=['start_time_DI', 'last_event_DI'])
df.head()


,index,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
0,0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,NaN,1.0
1,1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,NaN,0,2012-10-15,NaT,NaN,9.0,NaN,1.0,0,NaN,1.0
2,2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,NaN,1.0
3,3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-09-17,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0
4,4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0


In [3]:
# 원본데이터 결측치 개수, 비율
display(pd.DataFrame({
    'sum': df.isna().sum(),
    'ratio': df.isna().mean() * 100
}).sort_values('ratio', ascending=False).reset_index())

,index,sum,ratio
0,roles,641138,100.000000
1,incomplete_flag,540977,84.377622
2,nplay_video,457530,71.362172
3,nchapters,258753,40.358394
4,nevents,199151,31.062111
5,last_event_DI,178954,27.911932
6,ndays_act,162743,25.383459
7,LoE_DI,106008,16.534350
8,YoB,96605,15.067739
9,gender,86806,13.539363


In [4]:
# 전처리용 데이터 셋 생성
pre = df.copy()

In [5]:
# 중복행 확인
len(pre.duplicated())

641138

In [6]:
# 의미없는 컬럼 제거
pre = pre.drop(columns=['index', 'roles'])

In [7]:
#데이터 형변환
# grade 숫자형으로 변환
pre['grade'] = pd.to_numeric(pre['grade'], errors='coerce')

In [8]:
# 파생컬럼 생성

# 학생들의 나이(age) & (age_segment)
pre['age'] = pre['start_time_DI'].dt.year - pre['YoB']

conditions = [
    pre['age'] >= 60,
    pre['age'] >= 50,
    pre['age'] >= 40,
    pre['age'] >= 30,
    pre['age'] >= 20,
]

choices = ['60s+', '50s', '40s', '30s', '20s']

pre['age_segment'] = np.select(conditions, choices, default='under 20')

# 퍼널 단계 컬럼(step): 각 학생 별 진행 단계
pre['step'] = np.select(
    [
        pre['certified'] ==1,
        pre['explored'] == 1,
        pre['viewed'] == 1,
        pre['registered'] == 1,
    ],
    [
        'Certified',
        'Explored',
        'Viewed',
        'Registered'
    ],
    default='None'
)

# Missing Flag 컬럼 생성
missing_col = [
    'nchapters', 
    'nevents', 
    'ndays_act', 
    'nplay_video', 
    'last_event_DI', 
    'age', 
    'grade']

for col in missing_col:
    pre[f'{col}_flag'] = pre[col].isna().astype(int)
    

# 학습 기간 (duration) 컬럼 생성
pre['duration'] = (pre['last_event_DI'] - pre['start_time_DI']).dt.days.astype(int, errors='ignore')

In [9]:
# 결측치 대체

# 성별 결측치(gender) : unknown 대체
pre['gender'] = pre['gender'].fillna('unknown')

# 학력 결측치(LoE_DI) : unknown 대체
pre['LoE_DI'] = pre['LoE_DI'].fillna('unknown')

# 탐색한 챕터 수 결측치(nchapters) : registered 단계일 때 0으로 대체
pre['nchapters'] = pre[pre['step']=='Registered']['nchapters'].fillna(0)

# 총 이벤트 발생 수 결측치(nevent) : registered 단계일 때 0으로 대체
pre['nevents'] = pre[pre['step']=='Registered']['nevents'].fillna(0)

# 활성 일수 결측치(ndays_act) : 0으로 대체
pre['ndays_act'] = pre['ndays_act'].fillna(0)

# 영상재생횟수(nplay_video) 결측치 : 논의중

# 마지막 이벤트 발생일 (last_event_DI) : 논의중

# 나이 (age) 결측치 : 논의중

# 성적(grade) 결측치 : 논의중




In [15]:
# 행제거
print('행 제거 작업 시작 전:')
print(pre.shape)

# 퍼널 논리적 오류 행 제거
funnel_mask1 = (pre['viewed'] == 0) & (pre['explored'] == 1)
funnel_mask2 = (pre['explored'] == 0) & (pre['certified'] == 1)
pre = pre[~funnel_mask1]
pre = pre[~funnel_mask2]

# durration 음수 행 제거
duration_mask = pre['duration'] < 0
pre = pre[~duration_mask]

# age 13세 미만 행 제거
age_mask = pre['age'] < 13
pre = pre[~age_mask]

# nchapters가 0인 viewed, explored, certified 행 제거
nchapter_mask = (pre['step'].isin(['Certified', 'Viewed', 'Explored'])) & (pre['nchapters']==0)
pre = pre[~nchapter_mask]

# 상시 개방된 강의 제거 
course_mask = (pre['course_id'] =='HarvardX/CS50x/2012') | (pre['course_id'] =='HarvardX/ER22x/2013_Spring') | (pre['course_id'] =='HarvardX/CB22x/2013_Spring')
pre = pre[~course_mask]

# incomplete_flag == 1 제외
pre = pre[pre['incomplete_flag'].isna()]

print('행 제거 작업 시작 후:')
print(pre.shape)

행 제거 작업 시작 전:
(641138, 29)


C:\Users\gmltk\AppData\Local\Temp\ipykernel_384192\2960777977.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pre = pre[~funnel_mask2]


행 제거 작업 시작 후:
(355438, 29)


In [ ]:
# 2차 컬럼 drop
pre = pre.drop(columns=['YoB', 'incomplete_flag'])

In [14]:
pre.isna().sum()

course_id                  0
userid_DI                  0
registered                 0
viewed                     0
explored                   0
certified                  0
final_cc_cname_DI          0
LoE_DI                     0
gender                     0
grade                  57400
start_time_DI              0
last_event_DI         178954
nevents               400270
ndays_act                  0
nplay_video           457530
nchapters             400270
nforum_posts               0
incomplete_flag       540977
age                    96605
age_segment                0
step                       0
nchapters_flag             0
nevents_flag               0
ndays_act_flag             0
nplay_video_flag           0
last_event_DI_flag         0
age_flag                   0
grade_flag                 0
duration              178954
dtype: int64

In [11]:
pre['nevents'].value_counts().sort_index()

nevents
0.0       123094
1.0        54932
2.0        29044
3.0        13182
4.0         6421
5.0         3704
6.0         2214
7.0         1520
8.0         1028
9.0          717
10.0         612
11.0         527
12.0         407
13.0         323
14.0         263
15.0         255
16.0         224
17.0         170
18.0         168
19.0         136
20.0         127
21.0          93
22.0         102
23.0          90
24.0          86
25.0          76
26.0          76
27.0          51
28.0          83
29.0          65
30.0          53
31.0          37
32.0          46
33.0          48
34.0          59
35.0          39
36.0          43
37.0          38
38.0          31
39.0          20
40.0          30
41.0          16
42.0          20
43.0          25
44.0          18
45.0          15
46.0          20
47.0          13
48.0          16
49.0          14
50.0          17
51.0          17
52.0          18
53.0          12
54.0          21
55.0          13
56.0          12
57.0           9
58.0  